#### Скачайте данные
Загрузите данные, выполнив код ниже.

In [27]:
# from google.colab import files
# uploaded = files.upload()

In [28]:
import pandas as pd

data = pd.read_csv("../data/raw/NetflixShows.csv", encoding='cp437', sep=';')
del data['ratingDescription'], data['user rating size']

In [29]:
data

,title,rating,ratingLevel,release year,user rating score
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0
...,...,...,...,...,...
995,The BFG,PG,"for action/peril, some scary moments and brief...",2016,97.0
996,The Secret Life of Pets,PG,for action and some rude humor,2016,NaN
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN
998,Beary Tales,TV-G,Suitable for all ages.,2013,NaN


#### Удалите из данных дубликаты.
- Почему они возникли?
- Много ли их? В каких группах их больше всего?

In [30]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              1000 non-null   str    
 1   rating             1000 non-null   str    
 2   ratingLevel        941 non-null    str    
 3   release year       1000 non-null   int64  
 4   user rating score  605 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 39.2 KB


In [31]:
data.nunique()

title                496
rating                13
ratingLevel           99
release year          35
user rating score     42
dtype: int64

Найдем количество полных дублей, и удалим их из датасета при наличии.

In [32]:
duplicate = data.duplicated()
duplicate.sum()

np.int64(500)

500 строк в датасете - полные дубли, удалим их. 

In [33]:
data = data.drop_duplicates()
data.shape

(500, 5)

In [34]:
data["title"].unique()

<StringArray>
[             'White Chicks',       'Lucky Number Slevin',
            'Grey's Anatomy',              'Prison Break',
     'How I Met Your Mother',              'Supernatural',
              'Breaking Bad',       'The Vampire Diaries',
          'The Walking Dead',       'Pretty Little Liars',
 ...
                  'Flicka 2',       'H2O: Just Add Water',
              'Dolphin Tale',                 'Step Dogs',
                'Mia and Me',           'Russell Madness',
 'Wiener Dog Internationals',                  'Pup Star',
          'Precious Puppies',               'Beary Tales']
Length: 496, dtype: str

Проверим дублирование названий шоу и какие данные отличаются у одинаковых шоу.  

In [36]:
title_duplicate = data["title"].value_counts()
title_duplicate = title_duplicate[title_duplicate > 1]
title_duplicate.shape

(4,)

После удаления полных дублей осталось 4 шоу с повторяющимися названиями.
Выведем эти строки и найдем какие колонки отличаются у одинаковых шоу. 

In [37]:
title_duplicate.info()

<class 'pandas.Series'>
Index: 4 entries, Skins to Goosebumps
Series name: count
Non-Null Count  Dtype
--------------  -----
4 non-null      int64
dtypes: int64(1)
memory usage: 64.0+ bytes


In [38]:
twice_show = data[data["title"].isin(title_duplicate.index)]
twice_show

,title,rating,ratingLevel,release year,user rating score
151,Skins,TV-MA,For mature audiences. May not be suitable for...,2013,NaN
167,Bordertown,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,86.0
181,Skins,TV-MA,NaN,2017,NaN
449,Bordertown,TV-MA,For mature audiences. May not be suitable for...,2016,NaN
504,Star Wars: The Clone Wars,PG,"sci-fi action violence throughout, brief langu...",2008,57.0
512,Star Wars: The Clone Wars,TV-PG,Parental guidance suggested. May not be suitab...,2014,93.0
568,Goosebumps,TV-Y7,Suitable for children ages 7 and older,1998,88.0
632,Goosebumps,PG,"scary and intense creature action and images, ...",2015,90.0


У шоу Skins, Star Wars: The Clone Wars, Goosebumps отличаются год выпуска значит это разные шоу, не дубли.  
У шоу Bordertown отличается рейтинг (14+) и (18+).??

С дублями в датасете разобрались, проверим датасет на количетсво незаполненных ячеек.

In [39]:
data.isnull().sum()

title                  0
rating                 0
ratingLevel           33
release year           0
user rating score    244
dtype: int64

ratingLevel это описание рейтинговой группы. Заполним пустые значения описанием рейтинга, из заполненых ячеек того же рейтинга.  
user rating score это оценка пользователей, можно заполнить средним значением всего датасета, либо средним значением группировки по возрастному рейтингу


Нужно получить описание возрастного рейтинга (ratingLevel) для каждого уникального рейтинга (rating)

In [40]:
data.nunique()

title                496
rating                13
ratingLevel           99
release year          35
user rating score     42
dtype: int64

Так как количество уникальных ячеек в rating и ratingLevel разное, возьмем самые часто встречающиеся описания для каждого рейтинга.

In [41]:
ratingLevels = data.groupby("rating")["ratingLevel"].agg(lambda x: x.mode().iloc[0])
ratingLevels

rating
G                   General Audiences. Suitable for all ages.
NR                             This movie has not been rated.
PG          Parental guidance suggested. May not be suitab...
PG-13       For some rude and suggestive material, and for...
R           Restricted. May be inappropriate for children ...
TV-14       Parents strongly cautioned. May be unsuitable ...
TV-G                                   Suitable for all ages.
TV-MA       For mature audiences.  May not be suitable for...
TV-PG       Parental guidance suggested. May not be suitab...
TV-Y                                   Suitable for all ages.
TV-Y7                  Suitable for children ages 7 and older
TV-Y7-FV    Suitable for children ages 7 and older.  Conte...
UR          This movie has not been rated. Intended for ad...
Name: ratingLevel, dtype: str

In [42]:
data["ratingLevel"] = data["ratingLevel"].fillna(data["rating"].map(ratingLevels))

Теперь заполним пользовательскую оценку, медианой по рейтингу. 

In [43]:
UserScore = data.groupby("rating")["user rating score"].agg(lambda x: x.median())
UserScore

rating
G           70.0
NR          77.0
PG          86.0
PG-13       68.0
R           79.0
TV-14       86.0
TV-G        74.0
TV-MA       89.0
TV-PG       88.0
TV-Y        75.5
TV-Y7       74.5
TV-Y7-FV    72.0
UR           NaN
Name: user rating score, dtype: float64

Для одного рейтинга видим пустое значение, заполним медианой по всему датасету.

In [44]:
UserScore = UserScore.fillna(0)
UserScore

rating
G           70.0
NR          77.0
PG          86.0
PG-13       68.0
R           79.0
TV-14       86.0
TV-G        74.0
TV-MA       89.0
TV-PG       88.0
TV-Y        75.5
TV-Y7       74.5
TV-Y7-FV    72.0
UR           0.0
Name: user rating score, dtype: float64

In [45]:
# data["user rating score"] = data["user rating score"].fillna(data["rating"].map(UserScore))

In [46]:
data.isnull().sum()

title                  0
rating                 0
ratingLevel            0
release year           0
user rating score    244
dtype: int64

Добавим новый признак, основанный от рейтинга. Объединим признаки рейтинга в группы пользователей (Kids, Teen, Adult, Family)

In [47]:
Groups = {

    'G': 'Family',
    'TV-G': 'Family',

    'PG': 'Parental Guidance',
    'TV-PG': 'Parental Guidance',

    'TV-Y': 'Kids',
    'TV-Y7': 'Kids',
    'TV-Y7-FV': 'Kids',

    'PG-13': 'Teen',
    'TV-14': 'Teen',

    'R': 'Adult',
    'TV-MA': 'Adult',

    'NR': 'Unrated',
    'UR': 'Unrated'
}

In [48]:
data["ageGroup"] = data["rating"].map(Groups)

In [49]:
data

,title,rating,ratingLevel,release year,user rating score,ageGroup
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance
...,...,...,...,...,...,...
989,Russell Madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance
993,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family
994,Pup Star,G,General Audiences. Suitable for all ages.,2016,NaN,Family
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,Family


In [50]:
data.to_csv('../data/prepairedData.csv', sep=";", index=False)

In [64]:
prep = pd.read_csv('../data/prepairedData.csv', sep=";")

In [65]:

external_data_1 = pd.read_csv("../data/external/Netflix TV Shows and Movies.csv", encoding='cp437', sep=',')

In [66]:

external_data_2 = pd.read_csv("../data/external/netflix_titles_nov_2019.csv", encoding='cp437', sep=',')

In [69]:
external_data_1.head(3)

,index,id,title,type,description,release_year,age_certification,runtime,imdb_id,imdb_score,imdb_votes
0,0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,113,tt0075314,8.3,795222.0
1,1,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,tt0071853,8.2,530877.0
2,2,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,tt0079470,8.0,392419.0


In [70]:
external_data_2.head(3)

,show_id,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,type
0,81193313,Chocolate,NaN,"Ha Ji-won, Yoon Kye-sang, Jang Seung-jo, Kang ...",South Korea,"November 30, 2019",2019,TV-14,1 Season,"International TV Shows, Korean TV Shows, Roman...",Brought together by meaningful meals in the pa...,TV Show
1,81197050,Guatemala: Heart of the Mayan World,"Luis Ara, Ignacio Jaunsolo",Christian Morales,NaN,"November 30, 2019",2019,TV-G,67 min,"Documentaries, International Movies","From Sierra de las Minas to Esquipulas, explor...",Movie
2,81213894,The Zoya Factor,Abhishek Sharma,"Sonam Kapoor, Dulquer Salmaan, Sanjay Kapoor, ...",India,"November 30, 2019",2019,TV-14,135 min,"Comedies, Dramas, International Movies",A goofy copywriter unwittingly convinces the I...,Movie


Будем соединять по title. Для этого нужно привести их к одному формату. Возможные проблемы:
- Различие lower case и upper case (как полностью так и отдельных слов)
- Неуместное применение пробелов: 
    - "Невидимые пробелы" по бокам конкретного title
    - Сдвоенные пробелы между словами (либо отсутсвие пробела)
- Знаки припинания, например если названия битые или где-то их интерпретируют со знаком, а где-то нет.
- Есть вероятность использования алфавитов разных языков в данных (например "с" - английская и "c" - русская; кнопка на клавиатуре одна, а значения разные)

Допускаем что название фильма кодируется комбинацей букв английского алфавита (а может быть и нет) в определенном порядке. 

In [ ]:
ALPHABET = set("abcdefghijklmnopqrstuvwxyz0123456789")

def prepare_title(title:str):
    if pd.isna(title):
        return pd.NA
    
    title = str(title).lower().strip()

    res_title = []
    for char in title:
        if char in ALPHABET:
            res_title.append(char)

    return "".join(res_title)

Проверим. Добавим нормализацию в вите нового признака: "normal_title".

In [77]:
prep['normal_title'] = prep['title'].apply(lambda x: prepare_title(x))
prep.head(3)

,title,rating,ratingLevel,release year,user rating score,ageGroup,normal_title
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen,whitechicks
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult,luckynumberslevin
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen,greysanatomy


Аналогичная нормализация на внешние датасеты.

In [78]:
external_data_1['normal_title'] = external_data_1['title'].apply(lambda x: prepare_title(x))
external_data_2['normal_title'] = external_data_2['title'].apply(lambda x: prepare_title(x))

Инфо по признакам даасетов, с которыми работаем.

In [87]:
import numpy as np

pd.DataFrame({
    "prep" : pd.Series(prep.columns),
    "external_data_1" : pd.Series(external_data_1.columns),
    "external_data_2" : pd.Series(external_data_2.columns)
}).fillna('-')

,prep,external_data_1,external_data_2
0,title,index,show_id
1,rating,id,title
2,ratingLevel,title,director
3,release year,type,cast
4,user rating score,description,country
5,ageGroup,release_year,date_added
6,normal_title,age_certification,release_year
7,-,runtime,rating
8,-,imdb_id,duration
9,-,imdb_score,listed_in


Однако нам встеречались примеры, когда title один а строки по сути разные. В этом случае надо действовать по комбинации title + release year, чтобы не терять сезоны сериалов (или, например, ремастеры фильмов)

In [ ]:
external_data_1[external_data_1['normal_title'].isin(prep['normal_title'])]["normal_title"].value_counts()

normal_title
deathnote                       2
love                            2
fearless                        2
teenagemutantninjaturtles       1
annie                           1
                               ..
amyschumertheleatherspecial     1
littleboxes                     1
deidralaneyrobatrain            1
buddythunderstruck              1
felipenetomylifemakesnosense    1
Name: count, Length: 148, dtype: int64

In [112]:
external_data_2[external_data_2['normal_title'].isin(prep['normal_title'])]["normal_title"].value_counts()

normal_title
limitless                                3
love                                     3
littlebabybumnurseryrhymefriends         2
charmed                                  2
lovesick                                 2
                                        ..
gossipgirl                               1
breakingbad                              1
dreamworksspookystoriesvolume2           1
dreamworksshreksswampstories             1
dreamworkshowtotrainyourdragonlegends    1
Name: count, Length: 244, dtype: int64

Подобные случае есть и в первом датасете и во втором. Строк немного, рассмотрим их подробнее.

In [115]:
title_duples_df1 = external_data_1[external_data_1['normal_title'].isin(prep['normal_title'])]["normal_title"].value_counts()
title_duples_df1 = title_duples_df1[title_duples_df1 > 1]
title_duples_names_df1 = list(title_duples_df1.index)

In [116]:
title_duples_df2 = external_data_2[external_data_2['normal_title'].isin(prep['normal_title'])]["normal_title"].value_counts()
title_duples_df2 = title_duples_df2[title_duples_df2 > 1]
title_duples_names_df2 = list(title_duples_df2.index)

In [117]:
prep[prep['normal_title'].isin(title_duples_names_df1 + title_duples_names_df2)]

,title,rating,ratingLevel,release year,user rating score,ageGroup,normal_title
12,Death Note,TV-14,Parents strongly cautioned. May be unsuitable ...,2006,77.0,Teen,deathnote
34,Fearless,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,NaN,Teen,fearless
69,Love,TV-MA,For mature audiences. May not be suitable for...,2017,NaN,Adult,love
99,Lovesick,TV-MA,For mature audiences. May not be suitable for...,2016,NaN,Adult,lovesick
119,Skins,TV-MA,For mature audiences. May not be suitable for...,2013,NaN,Adult,skins
135,Skins,TV-MA,For mature audiences. May not be suitable for...,2017,NaN,Adult,skins
210,Charmed,TV-PG,Parental guidance suggested. May not be suitab...,2005,90.0,Parental Guidance,charmed
250,Aquarius,TV-MA,For mature audiences. May not be suitable for...,2015,NaN,Adult,aquarius
253,Limitless,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,84.0,Teen,limitless
374,Little Baby Bum: Nursery Rhyme Friends,TV-Y,Suitable for all ages.,2016,NaN,Kids,littlebabybumnurseryrhymefriends


**Конфликтные моменты в exteranl_data_1**

In [118]:
external_data_1[external_data_1['normal_title'].isin(title_duples_names_df1)]

,index,id,title,type,description,release_year,age_certification,runtime,imdb_id,imdb_score,imdb_votes,normal_title
256,256,ts11313,DEATH NOTE,SHOW,Light Yagami is an ace student with great pros...,2006,TV-14,24,tt0877057,9.0,302147.0,deathnote
1220,1220,ts38511,Love,SHOW,Rebellious Mickey and good-natured Gus navigat...,2016,TV-MA,32,tt4061080,7.7,41362.0,love
1517,1517,ts38812,Fearless,SHOW,On a journey from Brazil to the Las Vegas cham...,2016,NaN,43,tt9556710,7.1,299.0,fearless
1945,1945,tm217228,Death Note,MOVIE,A young man comes to possess a supernatural no...,2017,NC-17,101,tt1241317,4.5,83519.0,deathnote
3880,3880,tm918962,Fearless,MOVIE,A teen gamer is forced to level up to full-tim...,2020,PG,89,tt8675288,4.9,1706.0,fearless
4072,4072,tm946277,Love,MOVIE,The story of a family and the various situatio...,2020,NaN,91,tt12573294,7.0,1485.0,love


Абсолютно точно заметно, что дубли из-за существования SHOW и MOVIE с одним названием. Также у них отличаются release_year. 
- Поэтому для объединения датасетов будем использовать составной ключ (normal_title + year)
- Важно подчеркнуть что merge именно left, по той причине, что исходный датасет - центральный в нашем анализе.

In [ ]:
prep_added_ext1 = prep.merge()

**Конфликтные моменты в exteranl_data_2**

In [121]:
external_data_2[external_data_2['normal_title'].isin(title_duples_names_df2)].groupby('title').count()

,show_id,director,cast,country,date_added,release_year,rating,duration,listed_in,description,type,normal_title
title,,,,,,,,,,,,
Aquarius,2,1,2,2,1,2,2,2,2,2,2,2
Charmed,2,0,2,2,1,2,2,2,2,2,2,2
DEATH NOTE,1,0,1,1,1,1,1,1,1,1,1,1
Death Note,1,1,1,1,1,1,1,1,1,1,1,1
Limitless,3,2,2,3,3,3,3,3,3,3,3,3
Little Baby Bum: Nursery Rhyme Friends,2,0,1,0,0,2,1,2,2,2,2,2
Love,3,2,3,3,2,3,3,3,3,3,3,3
Lovesick,2,0,2,1,1,2,2,2,2,2,2,2
Skins,2,1,2,2,1,2,2,2,2,2,2,2


In [ ]:
# external_data['Title'] = external_data['Title'].str.lower().str.strip()
# data['title'] = data['title'].str.lower().str.strip()

In [ ]:
# common = external_data['Title'].isin(data['title']).sum()
# print(common)

246
